In [1]:
import pandas as pd
import numpy as np
import re
import eurostat 

In [2]:
EU_COUNTRIES = [
    'BE', 'BG', 'CZ', 'DK', 'DE', 'EE', 'IE', 
    'EL', 'ES', 'FR', 'HR', 'IT', 'CY', 'LV',
    'LT', 'LU', 'HU', 'MT', 'NL', 'AT', 'PL', 
    'PT', 'RO', 'SI', 'SK', 'FI', 'SE'
]

EFTA_COUNTRIES = ['IS', 'LI', 'NO', 'CH']

EU_EFTA = EU_COUNTRIES + EFTA_COUNTRIES

In [3]:
def nace_section_or_nan(s: str) -> str | float:
    s = str(s).strip().upper()
    match = re.fullmatch(r'([A-U])', s)

    if match:
        return match.group(1)
    else:
        return np.nan

## AI adoption (2021-2024 with gap year 2022)

In [14]:
ain2 = eurostat.get_data_df('isoc_eb_ain2')
print(ain2)

       freq size_emp nace_r2           indic_is    unit geo\TIME_PERIOD  2021  \
0         A     GE10       C    E_AIX_CC1SIX_DA  PC_ENT              AT   NaN   
1         A     GE10       C    E_AIX_CC1SIX_DA  PC_ENT              BA   NaN   
2         A     GE10       C    E_AIX_CC1SIX_DA  PC_ENT              BE   NaN   
3         A     GE10       C    E_AIX_CC1SIX_DA  PC_ENT              BG   NaN   
4         A     GE10       C    E_AIX_CC1SIX_DA  PC_ENT              CY   NaN   
...     ...      ...     ...                ...     ...             ...   ...   
236938    A     GE10    S951  E_DI3_VLO_AI_TANY  PC_ENT              RO   NaN   
236939    A     GE10    S951  E_DI3_VLO_AI_TANY  PC_ENT              RS   NaN   
236940    A     GE10    S951  E_DI3_VLO_AI_TANY  PC_ENT              SE   NaN   
236941    A     GE10    S951  E_DI3_VLO_AI_TANY  PC_ENT              SI   NaN   
236942    A     GE10    S951  E_DI3_VLO_AI_TANY  PC_ENT              SK   NaN   

         2023  2024   2025 

In [15]:
ain2.rename(columns={"geo\TIME_PERIOD": "geo"}, inplace=True)
ain2.describe()

<>:1: SyntaxWarning: invalid escape sequence '\T'
<>:1: SyntaxWarning: invalid escape sequence '\T'
C:\Users\ydmar\AppData\Local\Temp\ipykernel_25004\3075181159.py:1: SyntaxWarning: invalid escape sequence '\T'
  ain2.rename(columns={"geo\TIME_PERIOD": "geo"}, inplace=True)


,2021,2023,2024,2025
count,109575.000000,162572.000000,137079.000000,170512.000000
mean,10.853172,14.620912,14.050440,18.020146
std,21.003851,22.897424,20.980857,22.640605
min,0.000000,0.000000,0.000000,0.000000
25%,0.490000,1.200000,1.710000,3.100000
50%,2.410000,3.990000,5.240000,8.460000
75%,8.295000,16.250000,16.030000,22.800000
max,100.000000,100.000000,100.000000,100.000000


In [16]:
ain2.describe(include=["object", "bool"])

,freq,size_emp,nace_r2,indic_is,unit,geo
count,236943,236943,236943,236943,236943,236943
unique,1,1,50,63,6,36
top,A,GE10,C-E,E_AI_BINC,PC_ENT,PL
freq,236943,236943,4841,5359,95146,8138


In [17]:
# Filter the data 
# E_AI_TANY - Enterprises use at least one of the AI technologies

ai_adopt = ain2.copy()

ai_adopt = ai_adopt[
    (ai_adopt["indic_is"] == "E_AI_TANY") &
    (ai_adopt["unit"] == "PC_ENT") 
]

ai_adopt = ai_adopt.drop(columns=['indic_is','unit',  'freq', 'size_emp'])

ai_adopt = ai_adopt[ai_adopt['geo'].isin(EU_EFTA)]
print(ai_adopt)

       nace_r2 geo   2021   2023   2024   2025
3845         C  AT   9.61  12.31  22.71  32.54
3847         C  BE  10.42  15.31  23.24  39.80
3848         C  BG   2.88   2.55   4.35   5.25
3849         C  CY   2.05   3.81   2.98   4.26
3850         C  CZ   4.19   6.01   9.55  16.73
...        ...  ..    ...    ...    ...    ...
236006    S951  PT   7.58   8.82  21.94  23.33
236007    S951  RO   0.00   0.00   3.08   8.13
236009    S951  SE   6.67   6.67    NaN  43.06
236010    S951  SI   0.00    NaN  19.22    NaN
236011    S951  SK   0.00   0.00   9.09  14.94

[1396 rows x 6 columns]


In [18]:
ai_adopt['nace_r2_1d'] = ai_adopt['nace_r2'].map(nace_section_or_nan)
print(ai_adopt)

       nace_r2 geo   2021   2023   2024   2025 nace_r2_1d
3845         C  AT   9.61  12.31  22.71  32.54          C
3847         C  BE  10.42  15.31  23.24  39.80          C
3848         C  BG   2.88   2.55   4.35   5.25          C
3849         C  CY   2.05   3.81   2.98   4.26          C
3850         C  CZ   4.19   6.01   9.55  16.73          C
...        ...  ..    ...    ...    ...    ...        ...
236006    S951  PT   7.58   8.82  21.94  23.33        NaN
236007    S951  RO   0.00   0.00   3.08   8.13        NaN
236009    S951  SE   6.67   6.67    NaN  43.06        NaN
236010    S951  SI   0.00    NaN  19.22    NaN        NaN
236011    S951  SK   0.00   0.00   9.09  14.94        NaN

[1396 rows x 7 columns]


In [19]:
ai_adopt = ai_adopt.dropna()
print(pd.unique(ai_adopt['nace_r2']))
print(ai_adopt)

['C' 'D' 'E' 'F' 'G' 'H' 'I' 'J' 'L' 'M' 'N']
       nace_r2 geo   2021   2023   2024   2025 nace_r2_1d
3845         C  AT   9.61  12.31  22.71  32.54          C
3847         C  BE  10.42  15.31  23.24  39.80          C
3848         C  BG   2.88   2.55   4.35   5.25          C
3849         C  CY   2.05   3.81   2.98   4.26          C
3850         C  CZ   4.19   6.01   9.55  16.73          C
...        ...  ..    ...    ...    ...    ...        ...
222016       N  PT  10.81  10.29  12.91  11.06          N
222017       N  RO   3.04   1.57   5.30   5.04          N
222019       N  SE   8.58   8.30  21.63  31.61          N
222020       N  SI   9.02   3.22   7.65  20.20          N
222021       N  SK   8.60  10.16  18.37  19.79          N

[268 rows x 7 columns]


In [20]:
# Create the 2022 column by averaging 2021 and 2023
ai_adopt['2022'] = (ai_adopt['2021'] + ai_adopt['2023']) / 2

# Check the results
print(ai_adopt.head())

     nace_r2 geo   2021   2023   2024   2025 nace_r2_1d    2022
3845       C  AT   9.61  12.31  22.71  32.54          C  10.960
3847       C  BE  10.42  15.31  23.24  39.80          C  12.865
3848       C  BG   2.88   2.55   4.35   5.25          C   2.715
3849       C  CY   2.05   3.81   2.98   4.26          C   2.930
3850       C  CZ   4.19   6.01   9.55  16.73          C   5.100


In [21]:
ai_adopt = ai_adopt.drop(columns='nace_r2_1d')
cols_to_melt = ['2021', '2022', '2023', '2024', '2025']

df_ai_panel = ai_adopt.melt(
    id_vars=['geo', 'nace_r2'],      
    value_vars=cols_to_melt,        
    var_name='year_raw',             
    value_name='ai_adoption'        
)


df_ai_panel['year'] = df_ai_panel['year_raw'].str.extract(r'(\d+)').astype(int)
df_ai_panel.drop(columns=['year_raw'], inplace=True)
df_ai_panel = df_ai_panel.sort_values(by=['geo', 'nace_r2', 'year']).reset_index(drop=True)
print(df_ai_panel.head(10))

  geo nace_r2  ai_adoption  year
0  AT       C         9.61  2021
1  AT       C        10.96  2022
2  AT       C        12.31  2023
3  AT       C        22.71  2024
4  AT       C        32.54  2025
5  AT       F         3.12  2021
6  AT       F         3.70  2022
7  AT       F         4.28  2023
8  AT       F         7.35  2024
9  AT       F        14.89  2025


In [22]:
df_ai_panel.to_csv('data_panel/ai_adopt.csv', index = False)

## ICT training (2020, 2022, 2024)

In [23]:
train = eurostat.get_data_df('isoc_ske_ittn2')
print(train)


      freq size_emp nace_r2       indic_is         unit geo\TIME_PERIOD  \
0        A     GE10       C  E_ITSP2X_ITT2       PC_ENT              AT   
1        A     GE10       C  E_ITSP2X_ITT2       PC_ENT              BA   
2        A     GE10       C  E_ITSP2X_ITT2       PC_ENT              BE   
3        A     GE10       C  E_ITSP2X_ITT2       PC_ENT              BG   
4        A     GE10       C  E_ITSP2X_ITT2       PC_ENT              CY   
...    ...      ...     ...            ...          ...             ...   
16907    A     GE10    S951       E_ITUST2  PC_ENT_CUSE              SE   
16908    A     GE10    S951       E_ITUST2  PC_ENT_CUSE              SI   
16909    A     GE10    S951       E_ITUST2  PC_ENT_CUSE              SK   
16910    A     GE10    S951       E_ITUST2  PC_ENT_CUSE              TR   
16911    A     GE10    S951       E_ITUST2  PC_ENT_CUSE              UK   

        2012   2014   2015   2016   2017   2018   2019   2020  2022   2024  
0        NaN    NaN   

In [24]:
train.rename(columns={"geo\TIME_PERIOD": "geo"}, inplace=True)
train.drop(['2012', '2014', '2015', '2016', '2017', '2018', '2019'], axis='columns', inplace=True)
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16912 entries, 0 to 16911
Data columns (total 9 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   freq      16912 non-null  object 
 1   size_emp  16912 non-null  object 
 2   nace_r2   16912 non-null  object 
 3   indic_is  16912 non-null  object 
 4   unit      16912 non-null  object 
 5   geo       16912 non-null  object 
 6   2020      5499 non-null   float64
 7   2022      7278 non-null   float64
 8   2024      7335 non-null   float64
dtypes: float64(3), object(6)
memory usage: 1.2+ MB


<>:1: SyntaxWarning: invalid escape sequence '\T'
<>:1: SyntaxWarning: invalid escape sequence '\T'
C:\Users\ydmar\AppData\Local\Temp\ipykernel_25004\2930950165.py:1: SyntaxWarning: invalid escape sequence '\T'
  train.rename(columns={"geo\TIME_PERIOD": "geo"}, inplace=True)


In [25]:
# Filter the data 
train_new = train.copy()
train_new = train_new[
    (train_new['indic_is'] == 'E_ITT2') & # Enterprise provided training to their personnel to develop their ICT skills
    (train_new['unit'] == 'PC_ENT') # Percentage of enterprises 
]

train_new = train_new.drop(columns=['freq', 'size_emp', 'indic_is', 'unit'])
train_new = train_new[train_new['geo'].isin(EU_EFTA)]

print(train_new)

      nace_r2 geo   2020   2022   2024
239         C  AT  20.37  25.60  26.06
241         C  BE    NaN  33.89  40.12
242         C  BG   5.38   6.47   5.82
243         C  CY  19.69  20.51  17.41
244         C  CZ  27.60  23.76  28.35
...       ...  ..    ...    ...    ...
16783    S951  PT  63.29    NaN    NaN
16784    S951  RO  21.78  24.08  14.51
16786    S951  SE    NaN  53.33    NaN
16787    S951  SI    NaN    NaN  76.96
16788    S951  SK  18.33  23.08  18.18

[1459 rows x 5 columns]


In [26]:
train_new['nace_r2_1d'] = train_new['nace_r2'].map(nace_section_or_nan)
train_new.drop(columns=['nace_r2'], inplace=True) 
train_new.rename(columns={'nace_r2_1d' : 'nace_r2'}, inplace=True)
train_new = train_new.dropna()
print(train_new)

      geo   2020   2022   2024 nace_r2
239    AT  20.37  25.60  26.06       C
242    BG   5.38   6.47   5.82       C
243    CY  19.69  20.51  17.41       C
244    CZ  27.60  23.76  28.35       C
245    DE  26.58  27.82  28.73       C
...    ..    ...    ...    ...     ...
15591  PT  32.41  27.20  32.82       N
15592  RO   4.19   9.80  15.37       N
15594  SE  27.38  25.01  25.74       N
15595  SI  17.44  27.52  22.73       N
15596  SK  14.74   7.67  17.50       N

[182 rows x 5 columns]


In [27]:
train_new['2021'] = (train_new['2020'] + train_new['2022']) /2
train_new['2023'] = (train_new['2022'] + train_new['2024']) /2
print(train_new.head())

    geo   2020   2022   2024 nace_r2    2021    2023
239  AT  20.37  25.60  26.06       C  22.985  25.830
242  BG   5.38   6.47   5.82       C   5.925   6.145
243  CY  19.69  20.51  17.41       C  20.100  18.960
244  CZ  27.60  23.76  28.35       C  25.680  26.055
245  DE  26.58  27.82  28.73       C  27.200  28.275


In [28]:
cols_to_melt = ['2020', '2021', '2022', '2023', '2024']

df_train_panel = train_new.melt(
    id_vars=['geo', 'nace_r2'],      
    value_vars=cols_to_melt,         
    var_name='year_raw',            
    value_name='training_ict'        
)

df_train_panel['year'] = df_train_panel['year_raw'].str.extract(r'(\d+)').astype(int)
df_train_panel.drop(columns=['year_raw'], inplace=True)
df_train_panel = df_train_panel.sort_values(by=['geo', 'nace_r2', 'year']).reset_index(drop=True)

print(df_train_panel.head(10))

  geo nace_r2  training_ict  year
0  AT       C        20.370  2020
1  AT       C        22.985  2021
2  AT       C        25.600  2022
3  AT       C        25.830  2023
4  AT       C        26.060  2024
5  AT       F         9.650  2020
6  AT       F         8.280  2021
7  AT       F         6.910  2022
8  AT       F         8.010  2023
9  AT       F         9.110  2024


In [29]:
df_train_panel.to_csv('data_panel/train.csv', index = False)

## ICT specialists (2020, 2022, 2024)

In [30]:
ict_spec = eurostat.get_data_df('isoc_ske_itspen2')
ict_spec.rename(columns={"geo\TIME_PERIOD": "geo"}, inplace=True)
print(ict_spec)


<>:2: SyntaxWarning: invalid escape sequence '\T'
<>:2: SyntaxWarning: invalid escape sequence '\T'
C:\Users\ydmar\AppData\Local\Temp\ipykernel_25004\3175376399.py:2: SyntaxWarning: invalid escape sequence '\T'
  ict_spec.rename(columns={"geo\TIME_PERIOD": "geo"}, inplace=True)


     freq size_emp nace_r2 indic_is         unit geo   2012   2014    2015  \
0       A     GE10       C  E_ITSP2       PC_ENT  AL    NaN    NaN     NaN   
1       A     GE10       C  E_ITSP2       PC_ENT  AT  35.12  31.01   31.44   
2       A     GE10       C  E_ITSP2       PC_ENT  BA    NaN    NaN     NaN   
3       A     GE10       C  E_ITSP2       PC_ENT  BE  32.85    NaN     NaN   
4       A     GE10       C  E_ITSP2       PC_ENT  BG  10.34  17.20   16.76   
...   ...      ...     ...      ...          ...  ..    ...    ...     ...   
3410    A     GE10    S951  E_ITSP2  PC_ENT_CUSE  SE  75.00    NaN   72.73   
3411    A     GE10    S951  E_ITSP2  PC_ENT_CUSE  SI  71.43  83.33   83.33   
3412    A     GE10    S951  E_ITSP2  PC_ENT_CUSE  SK  41.96  57.50  100.00   
3413    A     GE10    S951  E_ITSP2  PC_ENT_CUSE  TR    NaN    NaN     NaN   
3414    A     GE10    S951  E_ITSP2  PC_ENT_CUSE  UK  66.67  65.38   78.18   

       2016   2017   2018   2019   2020   2022   2024  
0      

In [31]:
ict_spec.drop(['2012', '2014', '2015', '2016', '2017', '2018', '2019'], axis='columns', inplace=True)
ict_spec.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3415 entries, 0 to 3414
Data columns (total 9 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   freq      3415 non-null   object 
 1   size_emp  3415 non-null   object 
 2   nace_r2   3415 non-null   object 
 3   indic_is  3415 non-null   object 
 4   unit      3415 non-null   object 
 5   geo       3415 non-null   object 
 6   2020      1137 non-null   float64
 7   2022      1480 non-null   float64
 8   2024      1480 non-null   float64
dtypes: float64(3), object(6)
memory usage: 240.2+ KB


In [32]:
# Filter the data 
# E_ITSP2 - Enterprise employed ICT/IT specialists (reduced comparability with 2007)

ict_spec_new = ict_spec.copy()
ict_spec_new = ict_spec_new[
    (ict_spec_new['unit'] == 'PC_ENT') # Percentage of enterprises 
]

ict_spec_new = ict_spec_new.drop(columns=['freq', 'size_emp', 'indic_is', 'unit'])
ict_spec_new = ict_spec_new[ict_spec_new['geo'].isin(EU_EFTA)]

ict_spec_new['nace_r2_1d'] = ict_spec_new['nace_r2'].map(nace_section_or_nan)

ict_spec_new.drop(columns=['nace_r2'], inplace=True) 
ict_spec_new.rename(columns={'nace_r2_1d' : 'nace_r2'}, inplace=True)
ict_spec_new = ict_spec_new.dropna()
print(ict_spec_new)

     geo   2020   2022   2024 nace_r2
1     AT  28.74  28.37  30.02       C
4     BG  15.16  14.73  15.53       C
5     CY  14.23  16.29  14.93       C
6     CZ  21.66  21.42  23.05       C
7     DE  24.97  25.82  26.81       C
...   ..    ...    ...    ...     ...
3126  PT  29.51  29.67  29.37       N
3127  RO  14.31  11.35  15.56       N
3129  SE  17.58  17.64  11.41       N
3130  SI  10.39  14.76  13.80       N
3131  SK  16.90  13.37  13.68       N

[187 rows x 5 columns]


In [33]:
ict_spec_new['2021'] = (ict_spec_new['2020'] + ict_spec_new['2022']) /2
ict_spec_new['2023'] = (ict_spec_new['2022'] + ict_spec_new['2024']) /2
print(ict_spec_new.head())

  geo   2020   2022   2024 nace_r2    2021    2023
1  AT  28.74  28.37  30.02       C  28.555  29.195
4  BG  15.16  14.73  15.53       C  14.945  15.130
5  CY  14.23  16.29  14.93       C  15.260  15.610
6  CZ  21.66  21.42  23.05       C  21.540  22.235
7  DE  24.97  25.82  26.81       C  25.395  26.315


In [34]:
cols_to_melt = ['2020', '2021', '2022', '2023', '2024']

df_ict_panel = ict_spec_new.melt(
    id_vars=['geo', 'nace_r2'],     
    value_vars=cols_to_melt,       
    var_name='year_raw',            
    value_name='spec_ict'           
)

df_ict_panel['year'] = df_ict_panel['year_raw'].str.extract(r'(\d+)').astype(int)
df_ict_panel.drop(columns=['year_raw'], inplace=True)
df_ict_panel = df_ict_panel.sort_values(by=['geo', 'nace_r2', 'year']).reset_index(drop=True)
print(df_ict_panel.head(10))

  geo nace_r2  spec_ict  year
0  AT       C    28.740  2020
1  AT       C    28.555  2021
2  AT       C    28.370  2022
3  AT       C    29.195  2023
4  AT       C    30.020  2024
5  AT       F     8.240  2020
6  AT       F     8.635  2021
7  AT       F     9.030  2022
8  AT       F     8.075  2023
9  AT       F     7.120  2024


In [35]:
df_ict_panel.to_csv('data_panel/ict_spec.csv', index = False)

## Wages 

### Nominal wage (wage per hour in EUR)

In [36]:
lc = eurostat.get_data_df('lc_lci_lev')
lc.rename(columns={"geo\TIME_PERIOD": "geo"}, inplace=True)
lc.info()

<>:2: SyntaxWarning: invalid escape sequence '\T'
<>:2: SyntaxWarning: invalid escape sequence '\T'
C:\Users\ydmar\AppData\Local\Temp\ipykernel_25004\4158338232.py:2: SyntaxWarning: invalid escape sequence '\T'
  lc.rename(columns={"geo\TIME_PERIOD": "geo"}, inplace=True)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11051 entries, 0 to 11050
Data columns (total 13 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   freq      11051 non-null  object 
 1   unit      11051 non-null  object 
 2   lcstruct  11051 non-null  object 
 3   nace_r2   11051 non-null  object 
 4   geo       11051 non-null  object 
 5   2008      5462 non-null   float64
 6   2012      6986 non-null   float64
 7   2016      7510 non-null   float64
 8   2020      9767 non-null   float64
 9   2021      9635 non-null   float64
 10  2022      9639 non-null   float64
 11  2023      9643 non-null   float64
 12  2024      9349 non-null   float64
dtypes: float64(8), object(5)
memory usage: 1.1+ MB


In [37]:
lc.drop(['2008', '2012','2016'], axis='columns', inplace=True)

# Filter the data 
# per employee in full-time equivalents, per hour

wg = lc.copy()
wg = wg[
    (wg['unit'] == 'EUR') &
    (wg['lcstruct'] == 'D11') # Wages and Salaries (total)
]

wg = wg.drop(columns=['freq', 'unit', 'lcstruct'])
wg = wg[wg['geo'].isin(EU_EFTA)]

print(wg)

    nace_r2 geo  2020  2021  2022  2023  2024
1         B  AT  28.4  28.8  29.2  31.7  34.3
3         B  BE  31.3  31.7  34.0  36.7  37.6
4         B  BG   8.1   8.8  10.3  11.4  12.2
5         B  CH   NaN   NaN   NaN   NaN   NaN
6         B  CY  13.6  14.9  15.0  16.4  17.1
..      ...  ..   ...   ...   ...   ...   ...
879       S  PT  10.2  10.9  10.6  11.5  12.2
880       S  RO   5.4   5.5   6.6   7.2   8.5
882       S  SE  23.5  24.7  23.9  23.3  23.8
883       S  SI  15.0  15.3  15.9  18.2  19.5
884       S  SK   6.8   7.4   7.8   8.6   8.9

[635 rows x 7 columns]


In [38]:
wg['nace_r2_1d'] = wg['nace_r2'].map(nace_section_or_nan)

wg.drop(columns=['nace_r2'], inplace=True) 
wg.rename(columns={'nace_r2_1d' : 'nace_r2'}, inplace=True)

wg = wg.rename(columns={'2020': 'wg_n_2020',
                        '2021': 'wg_n_2021',
                        '2022' : 'wg_n_2022',
                        '2023' : 'wg_n_2023',
                        '2024' : 'wg_n_2024'})
print(wg)

    geo  wg_n_2020  wg_n_2021  wg_n_2022  wg_n_2023  wg_n_2024 nace_r2
1    AT       28.4       28.8       29.2       31.7       34.3       B
3    BE       31.3       31.7       34.0       36.7       37.6       B
4    BG        8.1        8.8       10.3       11.4       12.2       B
5    CH        NaN        NaN        NaN        NaN        NaN       B
6    CY       13.6       14.9       15.0       16.4       17.1       B
..   ..        ...        ...        ...        ...        ...     ...
879  PT       10.2       10.9       10.6       11.5       12.2       S
880  RO        5.4        5.5        6.6        7.2        8.5       S
882  SE       23.5       24.7       23.9       23.3       23.8       S
883  SI       15.0       15.3       15.9       18.2       19.5       S
884  SK        6.8        7.4        7.8        8.6        8.9       S

[635 rows x 7 columns]


### HICP (relative to 2015=100)

In [39]:
hicp = eurostat.get_data_df('prc_hicp_aind')
print(hicp)

      freq       unit          coicop geo\TIME_PERIOD  1996  1997  1998  1999  \
0        A     CID_EA  TOT_X_NRG_FOOD              AT   NaN   NaN   NaN   NaN   
1        A     CID_EA  TOT_X_NRG_FOOD              BE   NaN   NaN   NaN   NaN   
2        A     CID_EA  TOT_X_NRG_FOOD              BG   NaN   NaN   NaN   NaN   
3        A     CID_EA  TOT_X_NRG_FOOD              CY   NaN   NaN   NaN   NaN   
4        A     CID_EA  TOT_X_NRG_FOOD              CZ   NaN   NaN   NaN   NaN   
...    ...        ...             ...             ...   ...   ...   ...   ...   
35282    A  RCH_A_AVG       TOT_X_TBC              SI   NaN   NaN   NaN   NaN   
35283    A  RCH_A_AVG       TOT_X_TBC              SK   NaN   6.0   6.4  10.6   
35284    A  RCH_A_AVG       TOT_X_TBC              TR   NaN  85.0  82.6  61.4   
35285    A  RCH_A_AVG       TOT_X_TBC              UK   NaN   1.6   1.3   1.0   
35286    A  RCH_A_AVG       TOT_X_TBC              XK   NaN   NaN   NaN   NaN   

       2000  2001  ...  201

In [40]:
hicp.rename(columns={"geo\TIME_PERIOD": "geo"}, inplace=True)
years_to_drop = [str(year) for year in range(1996, 2020)]
hicp.drop(columns=years_to_drop, errors='ignore', inplace=True)
hicp.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 35287 entries, 0 to 35286
Data columns (total 10 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   freq    35287 non-null  object 
 1   unit    35287 non-null  object 
 2   coicop  35287 non-null  object 
 3   geo     35287 non-null  object 
 4   2020    32509 non-null  float64
 5   2021    32624 non-null  float64
 6   2022    32564 non-null  float64
 7   2023    32540 non-null  float64
 8   2024    32542 non-null  float64
 9   2025    32535 non-null  float64
dtypes: float64(6), object(4)
memory usage: 2.7+ MB


<>:1: SyntaxWarning: invalid escape sequence '\T'
<>:1: SyntaxWarning: invalid escape sequence '\T'
C:\Users\ydmar\AppData\Local\Temp\ipykernel_25004\908363276.py:1: SyntaxWarning: invalid escape sequence '\T'
  hicp.rename(columns={"geo\TIME_PERIOD": "geo"}, inplace=True)


In [41]:
# Filter the data 
# annual average index

hicp_inx = hicp.copy()
hicp_inx = hicp_inx.reset_index(drop=True)
hicp_inx = hicp_inx[
    (hicp_inx['unit'] == 'INX_A_AVG') &
    (hicp_inx['coicop'] == 'CP00') 
]

hicp_inx = hicp_inx.drop(columns=['freq', 'unit', 'coicop'])
hicp_inx = hicp_inx[hicp_inx['geo'].isin(EU_EFTA)]

print(hicp_inx)

    geo    2020    2021    2022    2023    2024    2025
233  AT  108.47  111.46  121.07  130.40  134.21  139.01
234  BE  108.23  111.71  123.26  126.07  131.52  135.49
235  BG  106.27  109.30  123.52  134.15  137.63  142.50
236  CH  100.56  101.04  103.74  106.10  107.25  107.36
237  CY   99.67  101.92  110.17  114.50  117.09  118.06
238  CZ  111.40  115.10  132.10  147.90  151.90  155.40
239  DE  105.80  109.20  118.70  125.90  129.00  131.90
240  DK  102.90  104.90  113.80  117.60  119.10  121.30
244  EE  109.80  114.72  137.03  149.52  155.10  162.57
246  EL  101.17  101.75  111.21  115.84  119.31  122.75
247  ES  103.91  107.04  115.95  119.89  123.33  126.65
251  FI  103.98  106.12  113.74  118.67  119.83  122.01
252  FR  105.50  107.68  114.04  120.50  123.29  124.43
253  HR  103.06  105.82  117.11  126.94  132.04  137.82
254  HU  113.15  119.04  137.22  160.59  166.56  173.96
255  IE  101.20  103.60  112.00  117.80  119.40  121.90
256  IS  103.06  106.84  112.96  121.96  127.47 

In [42]:
hicp_inx = hicp_inx.rename(columns={'2020': 'hicp_inx_2020',
                        '2021': 'hicp_inx_2021',
                        '2022' : 'hicp_inx_2022',
                        '2023' : 'hicp_inx_2023',
                        '2024' : 'hicp_inx_2024', 
                        '2025' : 'hicp_inx_2025'})

### Calculate real wages (2021-2024)

In [43]:
wage_countries = set(wg['geo'].unique())
hicp_countries = set(hicp_inx['geo'].unique())

missing_countries = wage_countries - hicp_countries

if len(missing_countries) > 0:
    print(f"WARNING: The following countries are in the Wage data but MISSING in HICP data:\n{missing_countries}")
else:
    print("All countries in Wage data have a matching HICP record.")

All countries in Wage data have a matching HICP record.


In [44]:
wg_merged = pd.merge(wg, hicp_inx, on='geo', how='left')

In [45]:
years = ['2020', '2021', '2022', '2023', '2024']

for year in years:
    # Define column names
    nom_col = f'wg_n_{year}'  
    hicp_col = f'hicp_inx_{year}'  
    real_col = f'wg_r_{year}'  
    
    # Apply formula
    wg_merged[real_col] = (wg_merged[nom_col] / wg_merged[hicp_col]) * 100

print(wg_merged)

    geo  wg_n_2020  wg_n_2021  wg_n_2022  wg_n_2023  wg_n_2024 nace_r2  \
0    AT       28.4       28.8       29.2       31.7       34.3       B   
1    BE       31.3       31.7       34.0       36.7       37.6       B   
2    BG        8.1        8.8       10.3       11.4       12.2       B   
3    CH        NaN        NaN        NaN        NaN        NaN       B   
4    CY       13.6       14.9       15.0       16.4       17.1       B   
..   ..        ...        ...        ...        ...        ...     ...   
630  PT       10.2       10.9       10.6       11.5       12.2       S   
631  RO        5.4        5.5        6.6        7.2        8.5       S   
632  SE       23.5       24.7       23.9       23.3       23.8       S   
633  SI       15.0       15.3       15.9       18.2       19.5       S   
634  SK        6.8        7.4        7.8        8.6        8.9       S   

     hicp_inx_2020  hicp_inx_2021  hicp_inx_2022  hicp_inx_2023  \
0           108.47         111.46         12

In [46]:
wg_merged = wg_merged.drop(['wg_n_2020', 'wg_n_2021', 'wg_n_2022', 'wg_n_2023', 'wg_n_2024',
                            'hicp_inx_2020', 'hicp_inx_2021', 'hicp_inx_2022', 'hicp_inx_2023', 'hicp_inx_2024'], axis='columns')
wg_merged = wg_merged.dropna()
print(wg_merged)

    geo nace_r2  hicp_inx_2025  wg_r_2020  wg_r_2021  wg_r_2022  wg_r_2023  \
0    AT       B         139.01  26.182355  25.838866  24.118279  24.309816   
1    BE       B         135.49  28.919893  28.377048  27.583969  29.110811   
2    BG       B         142.50   7.622095   8.051235   8.338731   8.497950   
4    CY       B         118.06  13.645029  14.619309  13.615322  14.323144   
5    CZ       B         155.40  10.053860  10.251955  10.068130  10.074375   
..   ..     ...            ...        ...        ...        ...        ...   
630  PT       S         124.83   9.847461  10.425634   9.378041   9.665490   
631  RO       S         160.06   4.879371   4.773891   5.113901   5.083310   
632  SE       S         132.35  21.834061  22.354964  20.018427  18.427713   
633  SI       S         131.04  14.310246  14.303076  13.596716  14.515872   
634  SK       S         149.23   6.269014   6.634986   6.237505   6.196412   

     wg_r_2024  
0    25.556963  
1    28.588808  
2     8.8643

In [47]:
wg_panel = wg_merged.melt(
    id_vars=['geo', 'nace_r2'],    
    value_vars=['wg_r_2020', 'wg_r_2021', 'wg_r_2022', 'wg_r_2023', 'wg_r_2024'], #
    var_name='year_raw',            
    value_name='real_wage'          
)

wg_panel['year'] = wg_panel['year_raw'].str.extract(r'(\d+)').astype(int)
wg_panel.drop(columns=['year_raw'], inplace=True)
wg_panel = wg_panel.sort_values(by=['geo', 'nace_r2', 'year']).reset_index(drop=True)

print("Panel Format Ready:")
print(wg_panel.head(10))

Panel Format Ready:
  geo nace_r2  real_wage  year
0  AT       B  26.182355  2020
1  AT       B  25.838866  2021
2  AT       B  24.118279  2022
3  AT       B  24.309816  2023
4  AT       B  25.556963  2024
5  AT       C  27.104268  2020
6  AT       C  26.736049  2021
7  AT       C  26.183200  2022
8  AT       C  26.073620  2023
9  AT       C  27.047165  2024


In [48]:
wg_panel.to_csv('data_panel/wage.csv', index = False)

### Calculate estimated wage 2025

## Productivity (2021-2024)

### GVA

In [49]:
gva = eurostat.get_data_df('nama_10_a64')

In [50]:
print(gva)

       freq        unit nace_r2 na_item geo\TIME_PERIOD  1975  1976  1977  \
0         A  CLV05_MEUR       A     B1G              AL   NaN   NaN   NaN   
1         A  CLV05_MEUR       A     B1G              AT   NaN   NaN   NaN   
2         A  CLV05_MEUR       A     B1G              BE   NaN   NaN   NaN   
3         A  CLV05_MEUR       A     B1G              BG   NaN   NaN   NaN   
4         A  CLV05_MEUR       A     B1G              CH   NaN   NaN   NaN   
...     ...         ...     ...     ...             ...   ...   ...   ...   
303102    A    PYP_MNAC     U99    P51C              PT   NaN   NaN   NaN   
303103    A    PYP_MNAC     U99    P51C              RO   NaN   NaN   NaN   
303104    A    PYP_MNAC     U99    P51C              RS   NaN   NaN   NaN   
303105    A    PYP_MNAC     U99    P51C              SI   NaN   NaN   NaN   
303106    A    PYP_MNAC     U99    P51C              SK   NaN   NaN   NaN   

        1978  1979  ...    2015    2016    2017    2018    2019    2020  \


In [51]:
gva.rename(columns={"geo\TIME_PERIOD": "geo"}, inplace=True)
gva.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 303107 entries, 0 to 303106
Data columns (total 55 columns):
 #   Column   Non-Null Count   Dtype  
---  ------   --------------   -----  
 0   freq     303107 non-null  object 
 1   unit     303107 non-null  object 
 2   nace_r2  303107 non-null  object 
 3   na_item  303107 non-null  object 
 4   geo      303107 non-null  object 
 5   1975     15231 non-null   float64
 6   1976     15599 non-null   float64
 7   1977     15603 non-null   float64
 8   1978     20366 non-null   float64
 9   1979     20957 non-null   float64
 10  1980     29453 non-null   float64
 11  1981     30546 non-null   float64
 12  1982     30549 non-null   float64
 13  1983     30549 non-null   float64
 14  1984     30553 non-null   float64
 15  1985     30556 non-null   float64
 16  1986     30560 non-null   float64
 17  1987     30563 non-null   float64
 18  1988     30563 non-null   float64
 19  1989     30563 non-null   float64
 20  1990     30563 non-null   

<>:1: SyntaxWarning: invalid escape sequence '\T'
<>:1: SyntaxWarning: invalid escape sequence '\T'
C:\Users\ydmar\AppData\Local\Temp\ipykernel_25004\1352719858.py:1: SyntaxWarning: invalid escape sequence '\T'
  gva.rename(columns={"geo\TIME_PERIOD": "geo"}, inplace=True)


In [52]:
print(gva)

       freq        unit nace_r2 na_item geo  1975  1976  1977  1978  1979  \
0         A  CLV05_MEUR       A     B1G  AL   NaN   NaN   NaN   NaN   NaN   
1         A  CLV05_MEUR       A     B1G  AT   NaN   NaN   NaN   NaN   NaN   
2         A  CLV05_MEUR       A     B1G  BE   NaN   NaN   NaN   NaN   NaN   
3         A  CLV05_MEUR       A     B1G  BG   NaN   NaN   NaN   NaN   NaN   
4         A  CLV05_MEUR       A     B1G  CH   NaN   NaN   NaN   NaN   NaN   
...     ...         ...     ...     ...  ..   ...   ...   ...   ...   ...   
303102    A    PYP_MNAC     U99    P51C  PT   NaN   NaN   NaN   NaN   NaN   
303103    A    PYP_MNAC     U99    P51C  RO   NaN   NaN   NaN   NaN   NaN   
303104    A    PYP_MNAC     U99    P51C  RS   NaN   NaN   NaN   NaN   NaN   
303105    A    PYP_MNAC     U99    P51C  SI   NaN   NaN   NaN   NaN   NaN   
303106    A    PYP_MNAC     U99    P51C  SK   NaN   NaN   NaN   NaN   NaN   

        ...    2015    2016    2017    2018    2019    2020    2021    2022

In [53]:
years_to_drop = [str(year) for year in range(1975, 2021)]
gva = gva.drop(columns=years_to_drop, errors='ignore')
gva['nace_r2_1d'] = gva['nace_r2'].map(nace_section_or_nan)
gva.drop(columns=['nace_r2'], inplace=True) 
gva.rename(columns={'nace_r2_1d' : 'nace_r2'}, inplace=True)
gva = gva[gva['geo'].isin(EU_EFTA)]

In [54]:
gva = gva[
    (gva['na_item'] == 'B1G') &         # Gross value added
    (gva['unit'] == 'CP_MEUR')        # Current prices, million EUR
]
print(gva)

       freq     unit na_item geo    2021    2022    2023    2024 nace_r2
125745    A  CP_MEUR     B1G  AT  4936.7  6050.9  5933.6  6011.2       A
125747    A  CP_MEUR     B1G  BE  3314.1  3768.8  4665.8  4910.3       A
125748    A  CP_MEUR     B1G  BG  3088.4  3194.7  2384.3  2463.1       A
125749    A  CP_MEUR     B1G  CH  4363.3  4878.7  5231.0  5463.8       A
125750    A  CP_MEUR     B1G  CY   385.8   351.6   374.7   401.6       A
...     ...      ...     ...  ..     ...     ...     ...     ...     ...
156491    A  CP_MEUR     B1G  PL     0.0     0.0     0.0     NaN     NaN
156492    A  CP_MEUR     B1G  PT     0.0     0.0     0.0     0.0     NaN
156493    A  CP_MEUR     B1G  RO     0.0     0.0     0.0     0.0     NaN
156495    A  CP_MEUR     B1G  SI     0.0     0.0     0.0     0.0     NaN
156496    A  CP_MEUR     B1G  SK     0.0     0.0     0.0     0.0     NaN

[2900 rows x 9 columns]


### Employees 

In [55]:
emp_df = eurostat.get_data_df('nama_10_a64_e')

emp = emp_df[
    (emp_df['na_item'] == 'EMP_DC') &     # Employment, domestic concept
    (emp_df['unit'] == 'THS_PER')       # Thousands of persons
]
print(emp)

      freq     unit nace_r2 na_item geo\TIME_PERIOD  1975  1976  1977  1978  \
34430    A  THS_PER       A  EMP_DC              AT   NaN   NaN   NaN   NaN   
34431    A  THS_PER       A  EMP_DC              BE   NaN   NaN   NaN   NaN   
34432    A  THS_PER       A  EMP_DC              BG   NaN   NaN   NaN   NaN   
34433    A  THS_PER       A  EMP_DC              CH   NaN   NaN   NaN   NaN   
34434    A  THS_PER       A  EMP_DC              CY   NaN   NaN   NaN   NaN   
...    ...      ...     ...     ...             ...   ...   ...   ...   ...   
45783    A  THS_PER     U99  EMP_DC              RO   NaN   NaN   NaN   NaN   
45784    A  THS_PER     U99  EMP_DC              SE   NaN   NaN   NaN   NaN   
45785    A  THS_PER     U99  EMP_DC              SI   NaN   NaN   NaN   NaN   
45786    A  THS_PER     U99  EMP_DC              SK   NaN   NaN   NaN   NaN   
45787    A  THS_PER     U99  EMP_DC              XK   NaN   NaN   NaN   NaN   

       1979  ...    2015    2016    2017    2018   

In [56]:
emp.rename(columns={"geo\TIME_PERIOD": "geo"}, inplace=True)
years_to_drop = [str(year) for year in range(1975, 2021)]
emp = emp.drop(columns=years_to_drop, errors='ignore')
emp['nace_r2_1d'] = emp['nace_r2'].map(nace_section_or_nan)
emp.drop(columns=['nace_r2'], inplace=True) 
emp.rename(columns={'nace_r2_1d' : 'nace_r2'}, inplace=True)
emp = emp[emp['geo'].isin(EU_EFTA)]
print(emp)

      freq     unit na_item geo    2021    2022    2023    2024 nace_r2
34430    A  THS_PER  EMP_DC  AT  155.35  149.98  138.12  136.50       A
34431    A  THS_PER  EMP_DC  BE   60.30   59.50   58.20   56.60       A
34432    A  THS_PER  EMP_DC  BG  532.61  521.84  517.02  488.91       A
34433    A  THS_PER  EMP_DC  CH  121.24  114.99  122.48  125.16       A
34434    A  THS_PER  EMP_DC  CY   15.80   15.99   16.09   16.32       A
...    ...      ...     ...  ..     ...     ...     ...     ...     ...
45782    A  THS_PER  EMP_DC  PT    0.00    0.00    0.00    0.00     NaN
45783    A  THS_PER  EMP_DC  RO    0.00    0.00    0.00    0.00     NaN
45784    A  THS_PER  EMP_DC  SE    0.00    0.00    0.00     NaN     NaN
45785    A  THS_PER  EMP_DC  SI    0.00    0.00    0.00    0.00     NaN
45786    A  THS_PER  EMP_DC  SK    0.00    0.00    0.00    0.00     NaN

[2943 rows x 9 columns]


<>:1: SyntaxWarning: invalid escape sequence '\T'
<>:1: SyntaxWarning: invalid escape sequence '\T'
C:\Users\ydmar\AppData\Local\Temp\ipykernel_25004\3286445458.py:1: SyntaxWarning: invalid escape sequence '\T'
  emp.rename(columns={"geo\TIME_PERIOD": "geo"}, inplace=True)
C:\Users\ydmar\AppData\Local\Temp\ipykernel_25004\3286445458.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  emp.rename(columns={"geo\TIME_PERIOD": "geo"}, inplace=True)


In [57]:
print(emp)

      freq     unit na_item geo    2021    2022    2023    2024 nace_r2
34430    A  THS_PER  EMP_DC  AT  155.35  149.98  138.12  136.50       A
34431    A  THS_PER  EMP_DC  BE   60.30   59.50   58.20   56.60       A
34432    A  THS_PER  EMP_DC  BG  532.61  521.84  517.02  488.91       A
34433    A  THS_PER  EMP_DC  CH  121.24  114.99  122.48  125.16       A
34434    A  THS_PER  EMP_DC  CY   15.80   15.99   16.09   16.32       A
...    ...      ...     ...  ..     ...     ...     ...     ...     ...
45782    A  THS_PER  EMP_DC  PT    0.00    0.00    0.00    0.00     NaN
45783    A  THS_PER  EMP_DC  RO    0.00    0.00    0.00    0.00     NaN
45784    A  THS_PER  EMP_DC  SE    0.00    0.00    0.00     NaN     NaN
45785    A  THS_PER  EMP_DC  SI    0.00    0.00    0.00    0.00     NaN
45786    A  THS_PER  EMP_DC  SK    0.00    0.00    0.00    0.00     NaN

[2943 rows x 9 columns]


### Productivity (calc)

In [58]:
gva = gva[['geo', 'nace_r2', '2021', '2022', '2023', '2024']].copy()
emp = emp[['geo', 'nace_r2', '2021', '2022', '2023', '2024']].copy()

gva = gva.rename(columns={str(y): f'gva_{y}' for y in [2021, 2022, 2023, 2024]})
emp = emp.rename(columns={str(y): f'emp_{y}' for y in [2021, 2022, 2023, 2024]})

prod = gva.merge(emp, on=['geo', 'nace_r2'], how='inner')

for y in [2021, 2022, 2023, 2024]:
    prod[f'gva_{y}'] = pd.to_numeric(prod[f'gva_{y}'], errors='coerce')
    prod[f'emp_{y}'] = pd.to_numeric(prod[f'emp_{y}'], errors='coerce')

print(prod)

       geo nace_r2  gva_2021  gva_2022  gva_2023  gva_2024  emp_2021  \
0       AT       A    4936.7    6050.9    5933.6    6011.2    155.35   
1       BE       A    3314.1    3768.8    4665.8    4910.3     60.30   
2       BG       A    3088.4    3194.7    2384.3    2463.1    532.61   
3       CH       A    4363.3    4878.7    5231.0    5463.8    121.24   
4       CY       A     385.8     351.6     374.7     401.6     15.80   
...     ..     ...       ...       ...       ...       ...       ...   
167970  SK     NaN       0.0       0.0       0.0       0.0     15.16   
167971  SK     NaN       0.0       0.0       0.0       0.0      4.33   
167972  SK     NaN       0.0       0.0       0.0       0.0     20.58   
167973  SK     NaN       0.0       0.0       0.0       0.0   2385.12   
167974  SK     NaN       0.0       0.0       0.0       0.0      0.00   

        emp_2022  emp_2023  emp_2024  
0         149.98    138.12    136.50  
1          59.50     58.20     56.60  
2         521.84  

In [59]:
# productivity (EUR per employee) for each year
for y in [2021, 2022, 2023, 2024]:
    gva_col = f'gva_{y}'    # million EUR
    emp_col = f'emp_{y}'    # thousand persons
    prod_col = f'prod_{y}'  # EUR per employee

    prod[prod_col] = (
        prod[gva_col] * 1_000_000    # EUR
    ) / (
        prod[emp_col] * 1_000        # persons
    )

    # Clean impossible values
    prod[prod_col] = prod[prod_col].replace([np.inf, -np.inf], np.nan)


print(prod.head())
print("\nCoverage:")
print("Countries:", prod['geo'].nunique())
print("NACE sectors:", prod['nace_r2'].unique())

  geo nace_r2  gva_2021  gva_2022  gva_2023  gva_2024  emp_2021  emp_2022  \
0  AT       A    4936.7    6050.9    5933.6    6011.2    155.35    149.98   
1  BE       A    3314.1    3768.8    4665.8    4910.3     60.30     59.50   
2  BG       A    3088.4    3194.7    2384.3    2463.1    532.61    521.84   
3  CH       A    4363.3    4878.7    5231.0    5463.8    121.24    114.99   
4  CY       A     385.8     351.6     374.7     401.6     15.80     15.99   

   emp_2023  emp_2024     prod_2021     prod_2022     prod_2023     prod_2024  
0    138.12    136.50  31777.920824  40344.712628  42959.745149  44038.095238  
1     58.20     56.60  54960.199005  63341.176471  80168.384880  86754.416961  
2    517.02    488.91   5798.614371   6121.991415   4611.620440   5037.941543  
3    122.48    125.16  35988.947542  42427.167580  42709.013717  43654.522212  
4     16.09     16.32  24417.721519  21988.742964  23287.756370  24607.843137  

Coverage:
Countries: 31
NACE sectors: ['A' nan 'B' 'C' '

In [60]:
prod_long = prod.melt(
    id_vars=['geo', 'nace_r2'],
    value_vars=[f'prod_{y}' for y in [2021, 2022, 2023, 2024]],
    var_name='year',
    value_name='productivity'
)
prod_long['year'] = prod_long['year'].str.replace('prod_', '').astype(int)
prod_long = prod_long.dropna(subset=['productivity'])
prod_long['productivity'] = round(prod_long['productivity'],2)
print(prod_long.head())

  geo nace_r2  year  productivity
0  AT       A  2021      31777.92
1  BE       A  2021      54960.20
2  BG       A  2021       5798.61
3  CH       A  2021      35988.95
4  CY       A  2021      24417.72


In [61]:
prod_long.to_csv('data_panel/product.csv', index = False)

## Firm size

In [62]:
fsi =  eurostat.get_data_df('sbs_sc_ovw') 
print(fsi)

        freq              indic_sbs nace_r2 size_emp geo\TIME_PERIOD    2021  \
0          A  AVG_EXPN_SAL_BEN_TEUR       B      0-9              AL    5.11   
1          A  AVG_EXPN_SAL_BEN_TEUR       B      0-9              AT   49.73   
2          A  AVG_EXPN_SAL_BEN_TEUR       B      0-9              BA    7.26   
3          A  AVG_EXPN_SAL_BEN_TEUR       B      0-9              BE   45.15   
4          A  AVG_EXPN_SAL_BEN_TEUR       B      0-9              BG   12.48   
...      ...                    ...     ...      ...             ...     ...   
1410270    A              WAGE_MEUR    S960    TOTAL              RO  199.03   
1410271    A              WAGE_MEUR    S960    TOTAL              RS   43.30   
1410272    A              WAGE_MEUR    S960    TOTAL              SE  863.44   
1410273    A              WAGE_MEUR    S960    TOTAL              SI   65.60   
1410274    A              WAGE_MEUR    S960    TOTAL              SK   46.87   

           2022    2023  2024  
0      

In [63]:
fsi.rename(columns={"geo\TIME_PERIOD": "geo"}, inplace=True)
fsi['nace_r2_1d'] = fsi['nace_r2'].map(nace_section_or_nan)
fsi.drop(columns=['nace_r2'], inplace=True) 
fsi.rename(columns={'nace_r2_1d' : 'nace_r2'}, inplace=True)
fsi = fsi[fsi['geo'].isin(EU_EFTA)]
fsi.info()

<>:1: SyntaxWarning: invalid escape sequence '\T'
<>:1: SyntaxWarning: invalid escape sequence '\T'
C:\Users\ydmar\AppData\Local\Temp\ipykernel_25004\2874492891.py:1: SyntaxWarning: invalid escape sequence '\T'
  fsi.rename(columns={"geo\TIME_PERIOD": "geo"}, inplace=True)


<class 'pandas.core.frame.DataFrame'>
Index: 1218936 entries, 1 to 1410274
Data columns (total 9 columns):
 #   Column     Non-Null Count    Dtype  
---  ------     --------------    -----  
 0   freq       1218936 non-null  object 
 1   indic_sbs  1218936 non-null  object 
 2   size_emp   1218936 non-null  object 
 3   geo        1218936 non-null  object 
 4   2021       899374 non-null   float64
 5   2022       929789 non-null   float64
 6   2023       925820 non-null   float64
 7   2024       301936 non-null   float64
 8   nace_r2    49568 non-null    object 
dtypes: float64(4), object(5)
memory usage: 93.0+ MB


In [64]:
# FSI Calculation (2021-2024) 

df_emp = fsi[fsi['indic_sbs'] == 'EMP_NR'].copy()
df_fsi = df_emp[df_emp['size_emp'].isin(['GE250', 'TOTAL'])].copy()

# Transform year columns into rows
df_melted = df_fsi.melt(
    id_vars=['geo', 'nace_r2', 'size_emp'],     
    value_vars=['2021', '2022', '2023', '2024'], 
    var_name='year_raw',
    value_name='emp_value'
)

df_melted['year'] = df_melted['year_raw'].str.extract(r'(\d+)').astype(int)

df_pivoted = df_melted.pivot_table(
    index=['geo', 'nace_r2', 'year'], 
    columns='size_emp',
    values='emp_value',
    aggfunc='sum'
).reset_index()

df_pivoted.rename(columns={'GE250': 'Emp_250_plus', 'TOTAL': 'Emp_Total'}, inplace=True)

df_pivoted['FSI'] = (df_pivoted['Emp_250_plus'] / df_pivoted['Emp_Total']) * 100

df_fsi_final = df_pivoted[['geo', 'nace_r2', 'year', 'FSI']].copy()

print("FSI Calculation Head (All Years):")
print(df_fsi_final.head())

FSI Calculation Head (All Years):
size_emp geo nace_r2  year        FSI
0         AT       B  2021   0.000000
1         AT       B  2022   0.000000
2         AT       B  2023   0.000000
3         AT       B  2024  39.214390
4         AT       C  2021  55.824158


In [65]:
df_fsi_final.to_csv('data_panel/fsi.csv', index = False)

## Education

In [66]:
educ =  eurostat.get_data_df('edat_lfs_9910') 

In [67]:
print(educ)

       freq unit nace_r2 isced11     age sex geo\TIME_PERIOD  2008  2009  \
0         A   PC       A   ED0-2  Y15-24   F              AT   NaN   NaN   
1         A   PC       A   ED0-2  Y15-24   F              BA   NaN   NaN   
2         A   PC       A   ED0-2  Y15-24   F              BE   NaN   NaN   
3         A   PC       A   ED0-2  Y15-24   F              BG   NaN   NaN   
4         A   PC       A   ED0-2  Y15-24   F              CH   NaN   NaN   
...     ...  ...     ...     ...     ...  ..             ...   ...   ...   
243325    A   PC       U   ED5-8  Y55-74   T              SE   NaN   NaN   
243326    A   PC       U   ED5-8  Y55-74   T              SI   NaN   NaN   
243327    A   PC       U   ED5-8  Y55-74   T              SK   NaN   NaN   
243328    A   PC       U   ED5-8  Y55-74   T              TR   NaN   NaN   
243329    A   PC       U   ED5-8  Y55-74   T              UK   NaN   NaN   

        2010  ...  2015  2016  2017  2018  2019  2020  2021  2022  2023  2024  
0      

In [68]:
educ.rename(columns={"geo\TIME_PERIOD": "geo"}, inplace=True)
educ.drop(['2008', '2009', '2010', '2011','2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019'], axis='columns', inplace=True)
educ.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 243330 entries, 0 to 243329
Data columns (total 12 columns):
 #   Column   Non-Null Count   Dtype  
---  ------   --------------   -----  
 0   freq     243330 non-null  object 
 1   unit     243330 non-null  object 
 2   nace_r2  243330 non-null  object 
 3   isced11  243330 non-null  object 
 4   age      243330 non-null  object 
 5   sex      243330 non-null  object 
 6   geo      243330 non-null  object 
 7   2020     114504 non-null  float64
 8   2021     157234 non-null  float64
 9   2022     158307 non-null  float64
 10  2023     156036 non-null  float64
 11  2024     156067 non-null  float64
dtypes: float64(5), object(7)
memory usage: 22.3+ MB


<>:1: SyntaxWarning: invalid escape sequence '\T'
<>:1: SyntaxWarning: invalid escape sequence '\T'
C:\Users\ydmar\AppData\Local\Temp\ipykernel_25004\84112933.py:1: SyntaxWarning: invalid escape sequence '\T'
  educ.rename(columns={"geo\TIME_PERIOD": "geo"}, inplace=True)


In [69]:
# Filter the data 
# unit - PC, percent

educ = educ.copy()
educ = educ[
    (educ['sex'] == 'T')& # for all genders
    (educ['age'] == 'Y18-69')& 
    (educ['isced11'] == 'ED5-8')   # 5-8: Tertiary education (levels 5-8)
]

educ = educ.drop(columns=['freq', 'unit', 'age', 'sex', 'isced11'])
educ = educ[educ['geo'].isin(EU_EFTA)]
educ = educ.dropna()

print(educ)

       nace_r2 geo  2020  2021  2022  2023  2024
9736         A  AT  30.4  22.2  24.4  22.6  33.8
9738         A  BE  19.8  28.7  36.2  36.0  41.1
9739         A  BG   8.1   7.5   6.7   8.6  12.7
9740         A  CH  21.7  23.4  22.9  23.9  29.5
9741         A  CY  15.9  21.5  21.4  18.9  17.3
...        ...  ..   ...   ...   ...   ...   ...
242302       U  CH  81.3  75.4  70.1  75.9  81.2
242303       U  CY  53.0  50.5  48.0  63.4  56.1
242313       U  FR  79.7  77.9  79.5  75.4  75.5
242318       U  IT  52.1  60.9  60.8  63.6  64.3
242320       U  LU  89.4  90.1  94.0  94.1  93.6

[573 rows x 7 columns]


In [70]:
education_long = educ.melt(
    id_vars=['nace_r2', 'geo'],
    value_vars=['2020', '2021', '2022', '2023', '2024'],
    var_name='year',
    value_name='tert_edu'
)

education_long['year'] = education_long['year'].astype(int)

education_long['tert_edu'] = pd.to_numeric(
    education_long['tert_edu'], 
    errors='coerce'
)

education_long = education_long[
    education_long['year'].isin([2021, 2022, 2023, 2024])
].copy()

print(education_long.head())

    nace_r2 geo  year  tert_edu
573       A  AT  2021      22.2
574       A  BE  2021      28.7
575       A  BG  2021       7.5
576       A  CH  2021      23.4
577       A  CY  2021      21.5


In [71]:
education_long.to_csv('data_panel/educ.csv', index = False)

## High skills

In [8]:
occup = eurostat.get_data_df('lfsa_eisn2')
print(occup)

      freq     age sex nace_r2 isco08     unit geo\TIME_PERIOD  2008  2009  \
0        A  Y20-64   F       A    NRP  THS_PER              CH   NaN   NaN   
1        A  Y20-64   F       A    NRP  THS_PER              DE   NaN   NaN   
2        A  Y20-64   F       A    NRP  THS_PER              DK   NaN   NaN   
3        A  Y20-64   F       A    NRP  THS_PER            EA21   NaN   NaN   
4        A  Y20-64   F       A    NRP  THS_PER       EU27_2020   NaN   NaN   
...    ...     ...  ..     ...    ...      ...             ...   ...   ...   
54815    A  Y_GE15   T       U  TOTAL  THS_PER              SE   NaN   NaN   
54816    A  Y_GE15   T       U  TOTAL  THS_PER              SI   NaN   NaN   
54817    A  Y_GE15   T       U  TOTAL  THS_PER              SK   NaN   NaN   
54818    A  Y_GE15   T       U  TOTAL  THS_PER              TR   NaN   5.3   
54819    A  Y_GE15   T       U  TOTAL  THS_PER              UK  12.2  43.4   

       2010  ...  2015  2016  2017  2018  2019  2020  2021  202

In [9]:
occup.rename(columns={"geo\TIME_PERIOD": "geo"}, inplace=True)
occup.drop(['2008', '2009', '2010', '2011','2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019'], axis='columns', inplace=True)
occup.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 54820 entries, 0 to 54819
Data columns (total 12 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   freq     54820 non-null  object 
 1   age      54820 non-null  object 
 2   sex      54820 non-null  object 
 3   nace_r2  54820 non-null  object 
 4   isco08   54820 non-null  object 
 5   unit     54820 non-null  object 
 6   geo      54820 non-null  object 
 7   2020     28538 non-null  float64
 8   2021     29340 non-null  float64
 9   2022     29531 non-null  float64
 10  2023     28977 non-null  float64
 11  2024     29064 non-null  float64
dtypes: float64(5), object(7)
memory usage: 5.0+ MB


<>:1: SyntaxWarning: invalid escape sequence '\T'
<>:1: SyntaxWarning: invalid escape sequence '\T'
C:\Users\ydmar\AppData\Local\Temp\ipykernel_25004\219748.py:1: SyntaxWarning: invalid escape sequence '\T'
  occup.rename(columns={"geo\TIME_PERIOD": "geo"}, inplace=True)


In [10]:
# Filter 
# unit - Ths_per, thousandpersons
occup = occup.copy()
occup = occup[
    (occup['sex'] == 'T')& # for all genders
    (occup['age'] == 'Y20-64') # From 20 to 64 years
]

occup = occup.drop(columns=['freq', 'unit', 'age', 'sex'])
occup = occup[occup['geo'].isin(EU_EFTA)]
print(occup)

      nace_r2 isco08 geo  2020  2021  2022  2023  2024
18036       A    NRP  CH   NaN   1.6   2.1   1.4   1.7
18037       A    NRP  DE   NaN   NaN   NaN   NaN   NaN
18038       A    NRP  DK   NaN   NaN   NaN   NaN   NaN
18041       A    NRP  FI   NaN   NaN   NaN   NaN   NaN
18042       A    NRP  FR   NaN   6.2   NaN   NaN   NaN
...       ...    ...  ..   ...   ...   ...   ...   ...
27369       U  TOTAL  PT   NaN   NaN   NaN   NaN   NaN
27370       U  TOTAL  RO   NaN   NaN   NaN   NaN   NaN
27372       U  TOTAL  SE   NaN   NaN   NaN   NaN   NaN
27373       U  TOTAL  SI   NaN   NaN   NaN   NaN   NaN
27374       U  TOTAL  SK   NaN   NaN   NaN   NaN   NaN

[7369 rows x 8 columns]


In [11]:
occup['nace_r2_1d'] = occup['nace_r2'].map(nace_section_or_nan)
occup.drop(columns=['nace_r2'], inplace=True) 
occup.rename(columns={'nace_r2_1d' : 'nace_r2'}, inplace=True)

occup = occup.dropna()
print(occup)

      isco08 geo  2020  2021  2022  2023  2024 nace_r2
18083    OC1  CZ   3.2   4.1   4.6   5.0   4.6       A
18089    OC1  ES  14.4  13.2  16.5  17.3  17.1       A
18092    OC1  FR  17.9  26.0  31.8  32.5  28.4       A
18093    OC1  HR   2.1   1.9   2.0   2.7   3.0       A
18094    OC1  HU   5.7   5.9   7.2   6.6   7.1       A
...      ...  ..   ...   ...   ...   ...   ...     ...
27347  TOTAL  DK   2.7   2.7   2.9   3.3   2.2       U
27351  TOTAL  ES   4.5   4.1   3.4   4.1   5.9       U
27354  TOTAL  FR  20.1  16.3  14.5  18.4  19.9       U
27359  TOTAL  IT  16.4  15.7  18.1  16.6  16.4       U
27361  TOTAL  LU  16.2  19.0  20.0  21.2  26.0       U

[3614 rows x 8 columns]


In [12]:
high_skill_codes = ['OC1', 'OC2', 'OC3']
total_code = 'TOTAL'
relevant_codes = high_skill_codes + [total_code]

occup_filtered = occup[occup['isco08'].isin(relevant_codes)].copy()

df_melted = occup_filtered.melt(
    id_vars=['geo', 'nace_r2', 'isco08'],          
    value_vars=['2020','2021', '2022', '2023', '2024'], 
    var_name='year_raw',
    value_name='emp_value'
)

df_melted['year'] = df_melted['year_raw'].str.extract(r'(\d+)').astype(int)

occup_pivoted = df_melted.pivot_table(
    index=['geo', 'nace_r2', 'year'], 
    columns='isco08',
    values='emp_value',
    aggfunc='sum'
).reset_index()


occup_pivoted.rename(columns={
    'OC1': 'OC1_Managers',
    'OC2': 'OC2_Professionals',
    'OC3': 'OC3_Technicians',
    'TOTAL': 'TOTAL_Occupied'
}, inplace=True)


occup_pivoted['total_high_skill'] = (
    occup_pivoted['OC1_Managers'] + 
    occup_pivoted['OC2_Professionals'] + 
    occup_pivoted['OC3_Technicians']
)



occup_pivoted['share_high_skill'] = (
    occup_pivoted['total_high_skill'] / occup_pivoted['TOTAL_Occupied']
) * 100



share_high_skill_df = occup_pivoted[['geo', 'nace_r2', 'year', 'share_high_skill']].copy()

share_high_skill_df = share_high_skill_df.dropna()

print("High Skill Share Calculation Head (Panel):")
print(share_high_skill_df.head())

High Skill Share Calculation Head (Panel):
isco08 geo nace_r2  year  share_high_skill
10      AT       C  2020         35.562831
11      AT       C  2021         36.579294
12      AT       C  2022         37.320574
13      AT       C  2023         38.711970
14      AT       C  2024         40.126825


In [13]:
share_high_skill_df.to_csv('data_panel/share_high_skill.csv', index = False)

## Data without GDP, Unempl, Infl variables

In [ ]:
# education_long
# df_fsi_final
# prod_long
# wg_panel
# df_ict_panel
# df_train_panel
# df_ai_panel
# share_high_skill_df

In [83]:
print(df_ai_panel)

     geo nace_r2  ai_adoption  year
0     AT       C         9.61  2021
1     AT       C        10.96  2022
2     AT       C        12.31  2023
3     AT       C        22.71  2024
4     AT       C        32.54  2025
...   ..     ...          ...   ...
1335  SK       N         8.60  2021
1336  SK       N         9.38  2022
1337  SK       N        10.16  2023
1338  SK       N        18.37  2024
1339  SK       N        19.79  2025

[1340 rows x 4 columns]


In [82]:
df_train_panel['training_ict'] = round(df_train_panel['training_ict'],2)
print(df_train_panel)

    geo nace_r2  training_ict  year
0    AT       C         20.37  2020
1    AT       C         22.98  2021
2    AT       C         25.60  2022
3    AT       C         25.83  2023
4    AT       C         26.06  2024
..   ..     ...           ...   ...
905  SK       N         14.74  2020
906  SK       N         11.20  2021
907  SK       N          7.67  2022
908  SK       N         12.58  2023
909  SK       N         17.50  2024

[910 rows x 4 columns]


In [80]:
df_ict_panel['spec_ict'] = round(df_ict_panel['spec_ict'],2)
print(df_ict_panel)

    geo nace_r2  spec_ict  year
0    AT       C     28.74  2020
1    AT       C     28.56  2021
2    AT       C     28.37  2022
3    AT       C     29.20  2023
4    AT       C     30.02  2024
..   ..     ...       ...   ...
930  SK       N     16.90  2020
931  SK       N     15.13  2021
932  SK       N     13.37  2022
933  SK       N     13.52  2023
934  SK       N     13.68  2024

[935 rows x 4 columns]


In [75]:
print(education_long)

     nace_r2 geo  year  tert_edu
573        A  AT  2021      22.2
574        A  BE  2021      28.7
575        A  BG  2021       7.5
576        A  CH  2021      23.4
577        A  CY  2021      21.5
...      ...  ..   ...       ...
2860       U  CH  2024      81.2
2861       U  CY  2024      56.1
2862       U  FR  2024      75.5
2863       U  IT  2024      64.3
2864       U  LU  2024      93.6

[2292 rows x 4 columns]


In [76]:
print(prod_long)

       geo nace_r2  year  productivity
0       AT       A  2021      31777.92
1       BE       A  2021      54960.20
2       BG       A  2021       5798.61
3       CH       A  2021      35988.95
4       CY       A  2021      24417.72
...     ..     ...   ...           ...
671894  SK     NaN  2024          0.00
671895  SK     NaN  2024          0.00
671896  SK     NaN  2024          0.00
671897  SK     NaN  2024          0.00
671898  SK     NaN  2024          0.00

[558217 rows x 4 columns]


In [78]:
wg_panel['real_wage'] = round(wg_panel['real_wage'],2)
print(wg_panel)

     geo nace_r2  real_wage  year
0     AT       B      26.18  2020
1     AT       B      25.84  2021
2     AT       B      24.12  2022
3     AT       B      24.31  2023
4     AT       B      25.56  2024
...   ..     ...        ...   ...
2200  SK       S       6.27  2020
2201  SK       S       6.63  2021
2202  SK       S       6.24  2022
2203  SK       S       6.20  2023
2204  SK       S       6.22  2024

[2205 rows x 4 columns]


In [74]:
df_fsi_final['FSI'] = round(df_fsi_final['FSI'],2)
print(df_fsi_final)

size_emp geo nace_r2  year    FSI
0         AT       B  2021   0.00
1         AT       B  2022   0.00
2         AT       B  2023   0.00
3         AT       B  2024  39.21
4         AT       C  2021  55.82
...       ..     ...   ...    ...
1915      SK       Q  2024  39.41
1916      SK       R  2021  19.07
1917      SK       R  2022   0.00
1918      SK       R  2023   0.00
1919      SK       R  2024   0.00

[1920 rows x 4 columns]


In [85]:
share_high_skill_df['share_high_skill'] = round(share_high_skill_df['share_high_skill'],2)
print(share_high_skill_df.head())

isco08 geo nace_r2  year  share_high_skill
10      AT       C  2020             35.56
11      AT       C  2021             36.58
12      AT       C  2022             37.32
13      AT       C  2023             38.71
14      AT       C  2024             40.13


In [86]:
# df_ai_panel
# df_train_panel
# df_ict_panel
# wg_panel
# prod_long
# df_fsi_final
# share_high_skill_df
# education_long

# Filter the base Wage panel to start from 2021
df_master = wg_panel.copy()
df_master = pd.merge(df_master, df_ai_panel, on=['geo', 'nace_r2', 'year'], how='left')
df_master = pd.merge(df_master, df_ict_panel, on=['geo', 'nace_r2', 'year'], how='left')
df_master = pd.merge(df_master, df_train_panel, on=['geo', 'nace_r2', 'year'], how='left')
df_master = pd.merge(df_master, prod_long, on=['geo', 'nace_r2', 'year'], how='left')
df_master = pd.merge(df_master, df_fsi_final, on=['geo', 'nace_r2', 'year'], how='left')
df_master = pd.merge(df_master, share_high_skill_df, on=['geo', 'nace_r2', 'year'], how='left')
df_master = pd.merge(df_master, education_long, on=['geo', 'nace_r2', 'year'], how='left')

print("\n--- MASTER PANEL DATASET (2021-2024) ---")
print(df_master['year'].value_counts().sort_index()) # Verifies only 2021, 2022, 2023, 2024 exist
print(df_master)


--- MASTER PANEL DATASET (2021-2024) ---
year
2020    441
2021    441
2022    441
2023    441
2024    441
Name: count, dtype: int64
     geo nace_r2  real_wage  year  ai_adoption  spec_ict  training_ict  \
0     AT       B      26.18  2020          NaN       NaN           NaN   
1     AT       B      25.84  2021          NaN       NaN           NaN   
2     AT       B      24.12  2022          NaN       NaN           NaN   
3     AT       B      24.31  2023          NaN       NaN           NaN   
4     AT       B      25.56  2024          NaN       NaN           NaN   
...   ..     ...        ...   ...          ...       ...           ...   
2200  SK       S       6.27  2020          NaN       NaN           NaN   
2201  SK       S       6.63  2021          NaN       NaN           NaN   
2202  SK       S       6.24  2022          NaN       NaN           NaN   
2203  SK       S       6.20  2023          NaN       NaN           NaN   
2204  SK       S       6.22  2024          NaN       

In [87]:
# Before dropna
print("Before dropna:")
print(f"Total rows: {len(df_master)}")
print(f"AI missing: {df_master['ai_adoption'].isna().sum()}")
print(f"Unique NACE: {df_master['nace_r2'].unique()}")

# After dropna
df_clean = df_master.dropna().copy()
print("\nAfter dropna:")
print(f"Total rows: {len(df_clean)}")
print(f"AI missing: {df_clean['ai_adoption'].isna().sum()}")
print(f"Unique NACE: {df_clean['nace_r2'].unique()}")


Before dropna:
Total rows: 2205
AI missing: 1249
Unique NACE: ['B' 'C' 'D' 'E' 'F' 'G' 'H' 'I' 'J' 'K' 'M' 'N' 'P' 'Q' 'R' 'S']

After dropna:
Total rows: 536
AI missing: 0
Unique NACE: ['C' 'F' 'G' 'H' 'N' 'I' 'J']


In [88]:
df_clean['log_prod'] = np.round(np.log(df_clean['productivity']), 2)
df_clean['log_wage'] = np.round(np.log(df_clean['real_wage']), 2)
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
Index: 536 entries, 6 to 2169
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   geo               536 non-null    object 
 1   nace_r2           536 non-null    object 
 2   real_wage         536 non-null    float64
 3   year              536 non-null    int32  
 4   ai_adoption       536 non-null    float64
 5   spec_ict          536 non-null    float64
 6   training_ict      536 non-null    float64
 7   productivity      536 non-null    float64
 8   FSI               536 non-null    float64
 9   share_high_skill  536 non-null    float64
 10  tert_edu          536 non-null    float64
 11  log_prod          536 non-null    float64
 12  log_wage          536 non-null    float64
dtypes: float64(10), int32(1), object(2)
memory usage: 56.5+ KB


In [89]:
print(df_clean)

     geo nace_r2  real_wage  year  ai_adoption  spec_ict  training_ict  \
6     AT       C      26.74  2021        9.610     28.56         22.98   
7     AT       C      26.18  2022       10.960     28.37         25.60   
8     AT       C      26.07  2023       12.310     29.20         25.83   
9     AT       C      27.05  2024       22.710     30.02         26.06   
21    AT       F      22.43  2021        3.120      8.64          8.28   
...   ..     ...        ...   ...          ...       ...           ...   
2154  SK       G       8.45  2024       11.650     15.19         19.87   
2166  SK       J      15.69  2021       18.190     70.76         55.96   
2167  SK       J      14.79  2022       19.905     71.88         53.54   
2168  SK       J      14.48  2023       21.620     70.80         58.16   
2169  SK       J      14.67  2024       29.190     69.72         62.79   

      productivity    FSI  share_high_skill  tert_edu  log_prod  log_wage  
6        102250.14  55.82          

In [90]:
df_clean.to_csv('data_panel/panel_master.csv', index = False )

## Data with macro metrics

### Unemployment rate

In [92]:
unempl = eurostat.get_data_df('tps00203')
print(unempl)

    freq     age     unit sex geo\TIME_PERIOD    2013    2014    2015    2016  \
0      A  Y15-74   PC_ACT   T              AT     5.7     6.0     6.1     6.5   
1      A  Y15-74   PC_ACT   T              BA     NaN     NaN     NaN     NaN   
2      A  Y15-74   PC_ACT   T              BE     8.6     8.7     8.7     7.9   
3      A  Y15-74   PC_ACT   T              BG    13.9    12.4    10.1     8.6   
4      A  Y15-74   PC_ACT   T              CH     4.8     4.9     4.8     5.0   
..   ...     ...      ...  ..             ...     ...     ...     ...     ...   
109    A  Y15-74  THS_PER   T              RS   732.0   627.0   570.0   506.0   
110    A  Y15-74  THS_PER   T              SE   414.0   414.0   391.0   371.0   
111    A  Y15-74  THS_PER   T              SI   101.0    98.0    90.0    79.0   
112    A  Y15-74  THS_PER   T              SK   394.0   366.0   323.0   274.0   
113    A  Y15-74  THS_PER   T              TR  2442.0  2843.0  3035.0  3308.0   

       2017    2018    2019

In [93]:
unempl.rename(columns={"geo\TIME_PERIOD": "geo"}, inplace=True)
unempl.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 114 entries, 0 to 113
Data columns (total 17 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   freq    114 non-null    object 
 1   age     114 non-null    object 
 2   unit    114 non-null    object 
 3   sex     114 non-null    object 
 4   geo     114 non-null    object 
 5   2013    111 non-null    float64
 6   2014    111 non-null    float64
 7   2015    111 non-null    float64
 8   2016    111 non-null    float64
 9   2017    111 non-null    float64
 10  2018    111 non-null    float64
 11  2019    111 non-null    float64
 12  2020    111 non-null    float64
 13  2021    111 non-null    float64
 14  2022    111 non-null    float64
 15  2023    111 non-null    float64
 16  2024    111 non-null    float64
dtypes: float64(12), object(5)
memory usage: 15.3+ KB


<>:1: SyntaxWarning: invalid escape sequence '\T'
<>:1: SyntaxWarning: invalid escape sequence '\T'
C:\Users\ydmar\AppData\Local\Temp\ipykernel_25004\1391666431.py:1: SyntaxWarning: invalid escape sequence '\T'
  unempl.rename(columns={"geo\TIME_PERIOD": "geo"}, inplace=True)


In [94]:
unempl.drop(['2013', '2014', '2015', '2016', '2017', '2018', '2019'], axis='columns', inplace=True)
print(unempl)

    freq     age     unit sex geo    2020    2021    2022    2023    2024
0      A  Y15-74   PC_ACT   T  AT     6.0     6.2     4.8     5.1     5.2
1      A  Y15-74   PC_ACT   T  BA     NaN    17.4    15.4    13.2    12.6
2      A  Y15-74   PC_ACT   T  BE     5.8     6.3     5.6     5.5     5.7
3      A  Y15-74   PC_ACT   T  BG     6.1     5.2     4.2     4.3     4.2
4      A  Y15-74   PC_ACT   T  CH     4.8     5.1     4.1     4.1     4.4
..   ...     ...      ...  ..  ..     ...     ...     ...     ...     ...
109    A  Y15-74  THS_PER   T  RS   290.0   344.0   296.0   296.0   272.0
110    A  Y15-74  THS_PER   T  SE   466.0   493.0   421.0   440.0   480.0
111    A  Y15-74  THS_PER   T  SI    51.0    48.0    41.0    38.0    38.0
112    A  Y15-74  THS_PER   T  SK   186.0   188.0   170.0   162.0   148.0
113    A  Y15-74  THS_PER   T  TR  4045.0  3915.0  3591.0  3274.0  3113.0

[114 rows x 10 columns]


In [ ]:
# Filter 
# Y15-74 default age class
# sex total is default


unempl_r = unempl.copy()
unempl_r = unempl_r[ 
    (unempl_r['unit'] == 'PC_ACT') # Percentage of population in the labour force
]

unempl_r = unempl_r.drop(columns=['freq', 'unit', 'age', 'sex'])
unempl_r = unempl_r[unempl_r['geo'].isin(EU_EFTA)]
print(unempl_r)

In [96]:
unempl_r = unempl_r.dropna()
print(unempl_r)

   geo  2020  2021  2022  2023  2024
0   AT   6.0   6.2   4.8   5.1   5.2
2   BE   5.8   6.3   5.6   5.5   5.7
3   BG   6.1   5.2   4.2   4.3   4.2
4   CH   4.8   5.1   4.1   4.1   4.4
5   CY   7.6   7.2   6.3   5.8   4.9
6   CZ   2.6   2.8   2.2   2.6   2.6
7   DE   3.6   3.6   3.1   3.1   3.4
8   DK   5.6   5.1   4.5   5.1   6.2
11  EE   6.9   6.2   5.6   6.4   7.6
12  EL  17.6  14.7  12.5  11.1  10.1
13  ES  15.5  14.9  13.0  12.2  11.4
15  FI   7.7   7.7   6.8   7.2   8.4
16  FR   8.0   7.9   7.3   7.3   7.4
17  HR   7.4   7.5   6.8   6.1   5.0
18  HU   4.1   4.0   3.6   4.1   4.5
19  IE   5.9   6.2   4.5   4.3   4.3
20  IS   5.5   6.1   3.8   3.5   3.6
21  IT   9.3   9.5   8.1   7.7   6.5
22  LT   8.5   7.1   6.0   6.9   7.1
23  LU   6.8   5.3   4.6   5.2   6.4
24  LV   8.1   7.6   6.9   6.5   6.9
27  MT   4.9   3.8   3.5   3.5   3.2
28  NL   4.9   4.2   3.5   3.6   3.7
29  NO   4.7   4.4   3.2   3.6   4.0
30  PL   3.2   3.4   2.9   2.8   2.9
31  PT   7.1   6.7   6.2   6.5   6.5
3

In [97]:
cols_to_melt = ['2020', '2021', '2022', '2023', '2024']

unempl_r_panel = unempl_r.melt(
    id_vars=['geo'],     
    value_vars=cols_to_melt,       
    var_name='year_raw',            
    value_name='unempl_r'           
)

unempl_r_panel['year'] = unempl_r_panel['year_raw'].str.extract(r'(\d+)').astype(int)
unempl_r_panel.drop(columns=['year_raw'], inplace=True)
unempl_r_panel = unempl_r_panel.sort_values(by=['geo', 'year']).reset_index(drop=True)
print(unempl_r_panel.head(10))

  geo  unempl_r  year
0  AT       6.0  2020
1  AT       6.2  2021
2  AT       4.8  2022
3  AT       5.1  2023
4  AT       5.2  2024
5  BE       5.8  2020
6  BE       6.3  2021
7  BE       5.6  2022
8  BE       5.5  2023
9  BE       5.7  2024


In [98]:
unempl_r_panel.to_csv('data_panel/unempl_r.csv', index = False)

### Inflation rate

In [100]:
infl = eurostat.get_data_df('tec00118')
infl.rename(columns={"geo\TIME_PERIOD": "geo"}, inplace=True)
infl.drop(['2014', '2015', '2016', '2017', '2018', '2019'], axis='columns', inplace=True)
infl.info()

<>:2: SyntaxWarning: invalid escape sequence '\T'
<>:2: SyntaxWarning: invalid escape sequence '\T'
C:\Users\ydmar\AppData\Local\Temp\ipykernel_25004\3800707276.py:2: SyntaxWarning: invalid escape sequence '\T'
  infl.rename(columns={"geo\TIME_PERIOD": "geo"}, inplace=True)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41 entries, 0 to 40
Data columns (total 10 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   freq      41 non-null     object 
 1   unit      41 non-null     object 
 2   coicop18  41 non-null     object 
 3   geo       41 non-null     object 
 4   2020      40 non-null     float64
 5   2021      40 non-null     float64
 6   2022      40 non-null     float64
 7   2023      40 non-null     float64
 8   2024      40 non-null     float64
 9   2025      39 non-null     float64
dtypes: float64(6), object(4)
memory usage: 3.3+ KB


In [ ]:
# Filter 
# unit RCH_A_AVG Annual average rate of change
# CP00 - All items HICP (harmonised index of consumer prices)

infl_r = infl.drop(columns=['freq', 'unit', 'coicop18'])
infl_r = infl_r[infl_r['geo'].isin(EU_EFTA)]
print(infl_r)

In [103]:
infl_r = infl_r.dropna()
print(infl_r)

   geo  2020  2021  2022  2023  2024  2025
1   AT   1.4   2.8   8.6   7.7   2.9   3.6
2   BE   0.4   3.2  10.3   2.3   4.3   3.0
3   BG   1.2   2.8  13.0   8.6   2.6   3.5
4   CH  -0.8   0.5   2.7   2.3   1.1   0.1
5   CY  -1.1   2.3   8.1   3.9   2.3   0.8
6   CZ   3.3   3.3  14.8  12.0   2.7   2.3
7   DE   0.4   3.2   8.7   6.0   2.5   2.3
8   DK   0.3   1.9   8.6   3.4   1.3   1.8
12  EE  -0.6   4.5  19.4   9.1   3.7   4.8
13  EL  -1.3   0.6   9.3   4.2   3.0   2.9
14  ES  -0.3   3.0   8.3   3.4   2.9   2.7
16  FI   0.4   2.1   7.2   4.3   1.0   1.8
17  FR   0.5   2.1   5.9   5.7   2.3   0.9
18  HR   0.0   2.7  10.7   8.4   4.0   4.4
19  HU   3.4   5.2  15.3  17.0   3.7   4.4
20  IE  -0.5   2.4   8.1   5.2   1.3   2.1
21  IS   1.2   3.7   5.7   8.0   4.5   3.7
22  IT  -0.2   2.0   8.7   5.9   1.1   1.6
23  LT   1.1   4.6  18.9   8.7   0.9   3.4
24  LU   0.0   3.5   8.2   2.9   2.3   2.5
25  LV   0.1   3.2  17.2   9.1   1.3   3.8
28  MT   0.8   0.7   6.1   5.6   2.4   2.4
29  NL   1.

In [104]:
infl_r_panel = infl_r.melt(
    id_vars=['geo'],     
    value_vars=cols_to_melt,       
    var_name='year_raw',            
    value_name='infl_r'           
)

infl_r_panel['year'] = infl_r_panel['year_raw'].str.extract(r'(\d+)').astype(int)
infl_r_panel.drop(columns=['year_raw'], inplace=True)
infl_r_panel = infl_r_panel.sort_values(by=['geo', 'year']).reset_index(drop=True)
print(infl_r_panel.head(10))

  geo  infl_r  year
0  AT     1.4  2020
1  AT     2.8  2021
2  AT     8.6  2022
3  AT     7.7  2023
4  AT     2.9  2024
5  BE     0.4  2020
6  BE     3.2  2021
7  BE    10.3  2022
8  BE     2.3  2023
9  BE     4.3  2024


In [105]:
infl_r_panel.to_csv('data_panel/infl_r.csv', index = False)

### GDP per capita

In [106]:
gdp_pc = eurostat.get_data_df('nama_10_pc')
print(gdp_pc)

     freq                      unit na_item geo\TIME_PERIOD  1975  1976  1977  \
0       A             CLV10_EUR_HAB    B1GQ              AL   NaN   NaN   NaN   
1       A             CLV10_EUR_HAB    B1GQ              AT   NaN   NaN   NaN   
2       A             CLV10_EUR_HAB    B1GQ              BE   NaN   NaN   NaN   
3       A             CLV10_EUR_HAB    B1GQ              BG   NaN   NaN   NaN   
4       A             CLV10_EUR_HAB    B1GQ              CH   NaN   NaN   NaN   
...   ...                       ...     ...             ...   ...   ...   ...   
4486    A  PC_EU27_2020_HAB_MPPS_CP     P41              SE   NaN   NaN   NaN   
4487    A  PC_EU27_2020_HAB_MPPS_CP     P41              SI   NaN   NaN   NaN   
4488    A  PC_EU27_2020_HAB_MPPS_CP     P41              SK   NaN   NaN   NaN   
4489    A  PC_EU27_2020_HAB_MPPS_CP     P41              TR   NaN   NaN   NaN   
4490    A  PC_EU27_2020_HAB_MPPS_CP     P41              UK   NaN   NaN   NaN   

      1978  1979  1980  ...

In [ ]:
cols_to_melt_upd = ['2020', '2021', '2022', '2023', '2024']
gdp_pc.rename(columns={"geo\TIME_PERIOD": "geo"}, inplace=True)
columns_to_drop = gdp_pc.columns[4:49]
gdp_pc.drop(columns=columns_to_drop, inplace=True)
gdp_pc.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4491 entries, 0 to 4490
Data columns (total 10 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   freq     4491 non-null   object 
 1   unit     4491 non-null   object 
 2   na_item  4491 non-null   object 
 3   geo      4491 non-null   object 
 4   2020     4383 non-null   float64
 5   2021     4383 non-null   float64
 6   2022     4374 non-null   float64
 7   2023     4266 non-null   float64
 8   2024     4245 non-null   float64
 9   2025     162 non-null    float64
dtypes: float64(6), object(4)
memory usage: 351.0+ KB


<>:1: SyntaxWarning: invalid escape sequence '\T'
<>:1: SyntaxWarning: invalid escape sequence '\T'
C:\Users\ydmar\AppData\Local\Temp\ipykernel_25004\871698340.py:1: SyntaxWarning: invalid escape sequence '\T'
  gdp_pc.rename(columns={"geo\TIME_PERIOD": "geo"}, inplace=True)


In [111]:
gdp_pc.drop(columns='2025', inplace=True)
gdp_pc = gdp_pc.dropna()
print(gdp_pc)

     freq                      unit na_item geo     2020     2021     2022  \
1       A             CLV10_EUR_HAB    B1GQ  AT  35480.0  37080.0  38620.0   
2       A             CLV10_EUR_HAB    B1GQ  BE  33990.0  35960.0  37090.0   
3       A             CLV10_EUR_HAB    B1GQ  BG   6800.0   7380.0   7730.0   
4       A             CLV10_EUR_HAB    B1GQ  CH  61140.0  64440.0  66160.0   
5       A             CLV10_EUR_HAB    B1GQ  CY  24190.0  26540.0  28210.0   
...   ...                       ...     ...  ..      ...      ...      ...   
4485    A  PC_EU27_2020_HAB_MPPS_CP     P41  RS     50.2     51.0     51.2   
4486    A  PC_EU27_2020_HAB_MPPS_CP     P41  SE    117.9    117.6    110.7   
4487    A  PC_EU27_2020_HAB_MPPS_CP     P41  SI     83.4     87.5     89.5   
4488    A  PC_EU27_2020_HAB_MPPS_CP     P41  SK     75.2     75.1     76.6   
4489    A  PC_EU27_2020_HAB_MPPS_CP     P41  TR     56.0     54.7     59.1   

         2023     2024  
1     37990.0  37550.0  
2     37400.0

In [112]:
# Filter 

gdp = gdp_pc.copy()
gdp = gdp[
    (gdp['na_item'] == 'B1GQ')& # Gross domestic product at market prices
    (gdp['unit'] == 'CP_EUR_HAB') # Current price, euro per capita
]


gdp = gdp.drop(columns=['freq', 'unit', 'na_item'])
gdp = gdp[gdp['geo'].isin(EU_EFTA)]
print(gdp)

     geo      2020      2021      2022      2023      2024
2608  AT   42650.0   45380.0   49640.0   52330.0   53830.0
2609  BE   40190.0   43680.0   48060.0   51140.0   52340.0
2610  BG    9440.0   10960.0   13310.0   14660.0   16260.0
2611  CH   76710.0   81610.0   92900.0   96280.0   99430.0
2612  CY   24630.0   27850.0   31560.0   33870.0   35730.0
2613  CZ   20980.0   23430.0   26670.0   29330.0   29440.0
2614  DE   42020.0   44910.0   48340.0   50660.0   51830.0
2615  DK   53540.0   58640.0   64430.0   62910.0   65650.0
2621  EE   20960.0   23650.0   27260.0   28080.0   28990.0
2622  EL   15660.0   17350.0   19570.0   21300.0   22480.0
2623  ES   23850.0   26090.0   28790.0   30980.0   32630.0
2625  FI   42740.0   44890.0   47890.0   48950.0   49100.0
2626  FR   34280.0   36920.0   38920.0   41340.0   42590.0
2627  HR   12980.0   15070.0   17540.0   20530.0   22200.0
2628  HU   14370.0   16090.0   17550.0   20560.0   21550.0
2629  IE   75820.0   88070.0  100140.0   99080.0  104510

In [113]:
gdp_panel = gdp.melt(
    id_vars=['geo'],     
    value_vars=cols_to_melt,       
    var_name='year_raw',            
    value_name='gdp'           
)

gdp_panel['year'] = gdp_panel['year_raw'].str.extract(r'(\d+)').astype(int)
gdp_panel.drop(columns=['year_raw'], inplace=True)
gdp_panel = gdp_panel.sort_values(by=['geo', 'year']).reset_index(drop=True)
print(gdp_panel.head(10))

  geo      gdp  year
0  AT  42650.0  2020
1  AT  45380.0  2021
2  AT  49640.0  2022
3  AT  52330.0  2023
4  AT  53830.0  2024
5  BE  40190.0  2020
6  BE  43680.0  2021
7  BE  48060.0  2022
8  BE  51140.0  2023
9  BE  52340.0  2024


In [114]:
gdp_panel.to_csv('data_panel/gdp.csv', index = False)